In [ ]:
# Extract dataset
!unzip tennis_court_det_dataset.zip

unzip:  cannot find or open tennis_court_det_dataset.zip, tennis_court_det_dataset.zip.zip or tennis_court_det_dataset.zip.ZIP.


# Implementation

In [3]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import json
import cv2
import numpy as np

In [4]:
# CPU or GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
class KeypointsDataset(Dataset):
    """
    This custom dataset class tells where images are, where labels are,
    and how to preprocess images.
    """
    def __init__(self, img_dir, data_file):
        self.img_dir = img_dir
        with open(data_file, "r") as f:
            self.data = json.load(f)
        
        self.transforms = transforms.Compose([
            transforms.ToPILImage(),        # to PIL image
            transforms.Resize((224, 224)),
            transforms.ToTensor(),      # To C x H x W
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    
    def __len__(self):
        return len(self.data)       # Dataset size.
    
    def __getitem__(self, idx):     # Load one example
        item = self.data[idx]
        img = cv2.imread(f"{self.img_dir}/{item['id']}.png")
        h,w = img.shape[:2]

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)      # Pytorch requires RGB images, OpenCV loads BGR images.
        img = self.transforms(img)
        kps = np.array(item['kps']).flatten()
        kps = kps.astype(np.float32)

        kps[::2] *= 224.0 / w # Adjust x coordinates since resized
        kps[1::2] *= 224.0 / h # Adjust y coordinates

        return img, kps

In [ ]:
# Create datasets and dataloaders
train_dataset = KeypointsDataset("data/images","data/data_train.json")
val_dataset = KeypointsDataset("data/images","data/data_val.json")

# 8 images per batch was chosen with shuffling to ensure that the model sees a variety of images in each batch.
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=True)

FileNotFoundError: [Errno 2] No such file or directory: 'data/data_train.json'

# Build model

In [ ]:
model = models.resnet50(pretrained=True)

# Replaces the last layer since 14 keypoints with x,y coordinates = 28 outputs
model.fc = torch.nn.Linear(model.fc.in_features, 14*2)

In [ ]:
# Move to GPU if available, otherwise CPU
model = model.to(device)

# Train model

In [ ]:
# Loss function and optimizer
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
epochs=20

In [ ]:
for epoch in range(epochs):
    for i, (imgs,kps) in enumerate(train_loader):

        # Move to GPU if available, otherwise CPU
        imgs = imgs.to(device)
        kps = kps.to(device)

        # Reset gradients, forward pass, compute loss, backward pass, and update weights
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, kps)
        loss.backward()
        optimizer.step()

        # Print loss every 10 iterations
        if i % 10 == 0:
            print(f"Epoch {epoch}, iter {i}, loss: {loss.item()}")

In [ ]:
# Save model weights to load later.
torch.save(model.state_dict(), "keypoints_model.pth")